# ReAct agent

Install Langchain dependencies

In [ ]:
%pip install -U langchain-nvidia-ai-endpoints langchain langchain-core langchain-community

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
#from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.agents import create_agent


Read credentials

In [ ]:
from google.colab import userdata
NVIDIA_API_KEY = userdata.get('apikey')
TAVILY_API_KEY = userdata.get('tavily')

Instantiate the model

In [ ]:
llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct",
    api_key = NVIDIA_API_KEY,
    temperature = 0.0,
    max_completion_tokens = 1024
    )

Tavily section

In [ ]:
%pip install tavily-python

In [ ]:
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

Function that uses Tavily to search for current weather at a specific city. This will be a tool for the model

In [ ]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    response = tavily_client.search(query=f"current weather in {city}", search_depth="basic")
    return response['results']

Create the agent

In [ ]:
SYSTEM_PROMPT = """You are an expert weather forecaster.
You have access to a tool:
- get_weather: use this to get the weather for a specific location"""

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=SYSTEM_PROMPT,
)

Let's try it out

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in Brisbane"}]}
)

In [ ]:
#import pprint
#pprint.pprint(response)
print(response["messages"][3].content)  # the fourth list item is AImessage
